# 16.16 聲音還沒全部產生，怎樣先播放第一段？


假設 Talker 正在逐步產生四個聲音碼 `[1,3,2,1]`。前兩碼已到，後兩碼還沒算完；播放器能否先收到一小段波形？關鍵是：現在交出去的樣本，之後不能因為新碼到來而被改寫。[12.18](https://birdhackor.github.io/tiny-perceptron-vlm/12.18.html)已把聲音碼接回 decoder（解碼器）；現在要讓它每收到一塊新碼，就真的算出一塊可固定下來的樣本。[16.3](https://birdhackor.github.io/tiny-perceptron-vlm/16.3.html)保存前文狀態、[16.13](https://birdhackor.github.io/tiny-perceptron-vlm/16.13.html)區分分塊與可見範圍的觀念，都用得上。

先用人工設定的小解碼器追數字。每碼查到一個值：碼0、1、2、3分別代表 `0、0.5、−0.25、1`。一碼會貢獻四個樣本，依次乘上 `[1,0.5,0.25,0.125]`；相鄰碼的起點卻只差兩個樣本。這個間距叫 **hop**。因此相鄰貢獻會重疊，重疊處相加。這也是轉置卷積能把較短的碼序列展成較長序列的一種具體運算。

碼1的貢獻是 `[0.5,0.25,0.125,0.0625]`。先交出前兩項，留下後兩項；碼3到來時，把它的前兩項 `[1,0.5]` 加上留下的尾部，得到 `[1.125,0.5625]`。新尾部則為 `[0.25,0.125]`。尾部是 **state（狀態）**：先前運算已產生、後面還要用的資訊。只保存已播放的樣本，無法完成這次相加。

下圖每塊包含兩個碼，所以一次輸出四個樣本。箭頭帶的是尚未完成的尾部，不是把已發出的首塊拿回來修改。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/omni-16.16-stream-tail.svg")))

這個解碼器只用當前碼與過去尾部，稱為**因果 decoder**。它的 lookahead（前看量，需要等多少未來碼）為0。下面 `push` 只處理本次收到的碼；第一個 `push` 執行時沒有取得後兩碼。整段的 `conv_transpose1d` 是另外建立的核對路徑。範例沒有訓練，也沒有正規化；若改用整段波形的平均值或標準差，即使卷積只看左邊，仍會透過統計量讀到未來。


In [ ]:
import torch
import torch.nn.functional as F

torch.set_num_threads(1)
table = torch.tensor([0.0, 0.5, -0.25, 1.0])
kernel = torch.tensor([1.0, 0.5, 0.25, 0.125])
hop = 2


class Decoder:
    def __init__(self):
        self.tail = torch.zeros(2)

    def push(self, ids):
        pieces = []
        for code in ids:
            contribution = table[code] * kernel
            # 只發出已完整的前 hop 個；後兩個留待下一碼相加。
            pieces.append(contribution[:hop] + self.tail)
            self.tail = contribution[hop:].clone()
        return torch.cat(pieces)

    def finish(self):
        # 本例定義 N 個碼恰好輸出 2N 個樣本；尾部丟棄。
        self.tail.zero_()


ids = torch.tensor([1, 3, 2, 1])
decoder = Decoder()
first = decoder.push(ids[:2])  # 此時沒有送入後兩碼
saved = decoder.tail.clone()
second = decoder.push(ids[2:])
streamed = torch.cat((first, second))
decoder.finish()

# 整段參考只用於核對，並不是串流路徑。
full = F.conv_transpose1d(table[ids][None, None], kernel[None, None], stride=hop).flatten()[: hop * len(ids)]
reset_second = Decoder().push(ids[2:])
future_changed = Decoder().push(torch.tensor([1, 3, 0, 3]))

print("首塊", first.tolist(), "保存尾部", saved.tolist())
print("第二塊", second.tolist())
print("逐塊／整段最大差", (streamed - full).abs().max().item())
print("重設後第二塊", reset_second.tolist())
print("重設造成最大差", (second - reset_second).abs().max().item())
print("改未來碼後舊prefix最大差", (first - future_changed[:4]).abs().max().item())
assert torch.equal(streamed, full)
assert not torch.equal(second, reset_second)
assert torch.equal(first, future_changed[:4])

關鍵輸出如下，數字都是這組人工參數的 CPU 運算結果：

```text
首塊 [0.5, 0.25, 1.125, 0.5625] 保存尾部 [0.25, 0.125]
第二塊 [0.0, 0.0, 0.4375, 0.21875]
逐塊／整段最大差 0.0
重設後第二塊 [-0.25, -0.125, 0.4375, 0.21875]
重設造成最大差 0.25
改未來碼後舊prefix最大差 0.0
```

第二塊開頭恰為0，是碼2的新貢獻 `−0.25` 抵銷尾部 `0.25`；清空 state 後便失去抵銷。另一個核對把未來兩碼改成 `[0,3]`，已發出的四個樣本仍相同。前者檢查有沒有接續過去，後者檢查有沒有偷讀未來。只把整段算完再切片，沒有展示第一個 `push` 能獨立完成。本例的結尾約定也寫在 `finish`：N碼只輸出2N個樣本，最後兩個尾部丟棄；新串流從零 state 起步，途中不重設。

這八個樣本只驗機制，沒有接播放器，不能拿運算秒數稱為實測串流延遲。真正量測時，可固定起點為服務端收到完整合法請求的時刻：**TTFA（Time to First Audio，首段音訊等待）**量到第一塊帶有取樣率、格式且播放器能接收的有效波形抵達客戶端；**全段完成時間**量到最後一塊抵達，兩者都包含各自範圍內的前處理、生成、解碼與傳輸。另報每塊的樣本數／取樣率，才知道它能播放多久。**穩態速度**則取首塊準備後的一段固定區間，本課統一用「計算秒數÷生成音訊秒數」表示 RTF（real-time factor，實時因子），小於1表示計算追得上播放；它不能回答首塊等了多久。

成熟系統也要逐一看等待邊界：[Qwen2.5-Omni §2.4](https://arxiv.org/html/2503.20215v1#S2.SS4)的DiT（Diffusion Transformer，擴散Transformer）把碼轉成mel頻譜，也就是各時間框的頻帶能量表示，再接波形生成器。此處先追蹤這個零件等哪些輸入，不推導擴散生成；它有2塊lookback（回看）與1塊lookahead。[Qwen3-Omni §2.5](https://arxiv.org/html/2509.17765v1#S2.SS5)才明說Code2Wav只讀左側上下文。零lookahead仍須等當前碼與實際運算。[EnCodec §3.1、§4.6](https://arxiv.org/pdf/2210.13438v1)也分開尾部state、正規化與初始等待；它採音訊秒數÷計算秒數的反向RTF定義，引用時要換算。

練習把兩次 `push` 改成每次只送一碼，將四次回傳值串起來。先預測八個樣本仍相同，再保留原來兩項因果核對。接著只在第二碼之前重建 `Decoder()`：哪兩個位置會先出錯？應是第2、3號樣本，因為丟掉第一碼留下的 `[0.125,0.0625]`；位置從0開始算。這說明每次送幾碼可以改，state 必須沿同一串流接續。
